# Week 1: Exploratory Data Analysis

**Telco Customer Churn Dataset**

This notebook presents a comprehensive exploratory data analysis following the CRISP-DM methodology.

---

## 1. Business Understanding

### Problem Statement
Predict which customers are likely to churn (cancel service) to enable proactive retention strategies.

### Success Criteria
- Identify key factors contributing to churn
- Build predictive model with >80% accuracy
- Provide actionable insights for retention

### Stakeholders
- Marketing Team: Target retention campaigns
- Customer Success: Identify at-risk customers
- Management: Reduce churn rate

---

## 2. Data Understanding

### Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

print("✓ Libraries imported successfully")

### Load Data

In [ ]:
# Load Telco Customer Churn dataset
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {', '.join(df.columns[:10])}...")  # Show first 10

### Data Dictionary Summary

| Category | Columns |
|----------|---------|
| Demographics | gender, SeniorCitizen, Partner, Dependents |
| Account | tenure, Contract, PaperlessBilling, PaymentMethod |
| Charges | MonthlyCharges, TotalCharges |
| Services | PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies |
| Target | Churn |

---

## 3. Exploratory Analysis

### 3.1 Descriptive Statistics

In [ ]:
# Numeric columns summary
print("NUMERIC COLUMNS:")
print("-"*60)
numeric_cols = ['tenure', 'MonthlyCharges', 'SeniorCitizen']
print(df[numeric_cols].describe().round(2))

# TotalCharges needs cleaning
print("\nNOTE: TotalCharges has empty strings that need cleaning")

### 3.2 Missing Values Analysis

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

print("MISSING VALUES:")
print("-"*60)
missing_df = pd.DataFrame({
    'Count': missing,
    'Percentage': missing_pct
})
print(missing_df[missing_df['Count'] > 0])

# Check for empty strings in TotalCharges
empty_strings = (df['TotalCharges'] == ' ').sum()
print(f"\nEmpty strings in TotalCharges: {empty_strings}")
print(f"These correspond to customers with tenure=0 (new customers)")

### 3.3 Target Variable Distribution

In [ ]:
# Churn distribution
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print("CHURN DISTRIBUTION:")
print("-"*60)
for churn_val in churn_counts.index:
    print(f"{churn_val}: {churn_counts[churn_val]:,} ({churn_pct[churn_val]:.1f}%)")

# Visualization
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='Churn', palette='Set2')
for i, v in enumerate(churn_counts):
    ax.text(i, v + 50, f"{v:,}\n({churn_pct.iloc[i]:.1f}%)", 
            ha='center', va='bottom', fontsize=11)
plt.title('Customer Churn Distribution', fontsize=14)
plt.xlabel('Churn')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../outputs/churn_distribution.png', dpi=300)
plt.show()

### 3.4 Key Relationships

#### Monthly Charges by Churn

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', palette='Set2')
plt.title('Monthly Charges by Churn Status', fontsize=14)
plt.ylabel('Monthly Charges ($)')
plt.tight_layout()
plt.savefig('../outputs/monthly_charges_by_churn.png', dpi=300)
plt.show()

# Statistics
print("\nMonthly Charges by Churn:")
print(df.groupby('Churn')['MonthlyCharges'].agg(['mean', 'median']).round(2))

#### Tenure Distribution by Churn

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='tenure', hue='Churn', bins=30, kde=True, palette='Set2')
plt.title('Tenure Distribution by Churn Status', fontsize=14)
plt.xlabel('Tenure (months)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../outputs/tenure_distribution.png', dpi=300)
plt.show()

# Statistics
print("\nTenure by Churn:")
print(df.groupby('Churn')['tenure'].agg(['mean', 'median']).round(2))

#### Churn by Contract Type

In [ ]:
plt.figure(figsize=(10, 6))
contract_churn = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
contract_churn.plot(kind='bar', color=['steelblue', 'coral'])
plt.title('Churn Rate by Contract Type', fontsize=14)
plt.xlabel('Contract Type')
plt.ylabel('Percentage')
plt.legend(title='Churn')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../outputs/churn_by_contract.png', dpi=300)
plt.show()

print("\nChurn Rate by Contract:")
print(contract_churn.round(1))

#### Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))

# Prepare numeric data
numeric_df = df[['tenure', 'MonthlyCharges', 'SeniorCitizen']].copy()
numeric_df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
numeric_df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Correlation matrix
corr = numeric_df.corr()

sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.2f')
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/correlation_heatmap.png', dpi=300)
plt.show()

print("\nCorrelations with Churn:")
print(corr['Churn'].sort_values(ascending=False).round(3))

---

## 4. Initial Findings

### Key Insights

1. **Churn Rate**: 26.5% of customers have churned, representing significant business impact

2. **Tenure Effect**: 
   - Customers who churn have lower average tenure (18 months vs 38 months)
   - New customers (tenure < 12 months) show higher churn risk

3. **Monthly Charges Impact**:
   - Churned customers have higher average monthly charges ($74 vs $61)
   - Price sensitivity appears to be a factor

4. **Contract Type**:
   - Month-to-month contracts: 42% churn rate
   - One year contracts: 11% churn rate
   - Two year contracts: 3% churn rate

5. **Service Patterns**:
   - Fiber optic internet customers have higher churn rates
   - Customers without tech support show higher churn

### Hypotheses Formed

1. Tenure is the strongest protective factor against churn
2. Higher monthly charges increase churn probability
3. Contract length significantly impacts retention
4. Service quality issues (fiber optic) may drive churn

---

## 5. Next Steps

### Data Preparation
1. Clean TotalCharges (convert to numeric, handle empty values)
2. Encode categorical variables
3. Create derived features (tenure groups, total services)
4. Handle class imbalance

### Modeling Approach
1. Establish baseline with Logistic Regression
2. Try ensemble methods (Random Forest, XGBoost)
3. Evaluate with appropriate metrics (Precision, Recall, F1, ROC-AUC)
4. Perform hyperparameter tuning

### Questions to Answer
1. Which features have the highest predictive power?
2. Can we identify high-risk customer segments?
3. What are the most actionable retention strategies?

---

## Appendix: Data Quality Summary

| Check | Result |
|-------|--------|
| Duplicate rows | 0 |
| Missing values (null) | 0 |
| Empty TotalCharges | 11 (0.16%) |
| Data types | Mixed (needs cleaning) |
| Target balance | 73.5% No, 26.5% Yes |